# Zero-shot Contrastive Schema Designs for GLiNER2

This notebook tests whether hand/rule-designed label schemas can improve Banking77 intent classification without using Banking77 train examples, train labels, external LLMs, fine-tuning, LoRA, paid APIs, or GLiNER2 internals.

It evaluates description-based labels, cluster-aware second passes, hierarchical grouping, and oracle analysis among zero-shot schema variants.


In [ ]:
PROJECT_NAME = "gliner2_zero_shot_contrastive_schema"
MODEL_NAME = "fastino/gliner2-base-v1"
DATASET_NAME = "mteb/banking77"
MODE = "small"  # smoke / small / full
SEED = 42
SMOKE_N_EXAMPLES = 20
SMALL_PER_LABEL = 5
FULL_USE_ALL_TEST = True
CONFIRM_FULL_RUN = False
FORCE_RERUN = False
OUTPUT_DIR = "/content/gliner2_schema_outputs"
ZERO_SHOT_SCHEMA_OUTPUT_SUBDIR = "zero_shot_contrastive_schema"

RUN_DESCRIPTION_ALL_LABELS = True
RUN_CLUSTER_SECOND_PASS = True
RUN_HIERARCHICAL_GLINER = True
RUN_ORACLE_ANALYSIS = True

if MODE == "full" and not CONFIRM_FULL_RUN:
    raise RuntimeError(
        "MODE='full' requires CONFIRM_FULL_RUN=True. This prevents accidental long runs."
    )


## Install and import dependencies


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

OUTPUT_PATH = Path(OUTPUT_DIR) / ZERO_SHOT_SCHEMA_OUTPUT_SUBDIR
PREDICTIONS_DIR = OUTPUT_PATH / "predictions"
TABLES_DIR = OUTPUT_PATH / "tables"
FIGURES_DIR = OUTPUT_PATH / "figures"
for path in [OUTPUT_PATH, PREDICTIONS_DIR, TABLES_DIR, FIGURES_DIR]:
    path.mkdir(parents=True, exist_ok=True)


def find_project_root():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/content"),
        Path("/content/GLiNER2-demo"),
        Path("/content/gliner2_schema_wording"),
        Path("/content/drive/MyDrive/GLiNER2-demo"),
        Path("/content/drive/MyDrive/gliner2_schema_wording"),
    ]
    for root in candidates:
        if (root / "requirements-colab.txt").exists() and (root / "src").exists():
            return root.resolve()
    for pattern in ["*/requirements-colab.txt", "*/*/requirements-colab.txt"]:
        for req_path in Path("/content").glob(pattern):
            root = req_path.parent
            if (root / "src").exists():
                return root.resolve()
    return None


PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None:
    repo_url = "https://github.com/daidai-su/GLiNER2-demo.git"
    clone_dir = Path("/content/GLiNER2-demo")
    print("Project files were not found in the runtime.")
    print(f"Trying to clone project files from {repo_url} ...")
    try:
        if not clone_dir.exists():
            subprocess.check_call(["git", "clone", "--depth", "1", repo_url, str(clone_dir)])
        PROJECT_ROOT = find_project_root()
    except Exception as exc:
        print(f"Automatic clone failed: {exc}")

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find project root. Upload or clone the full repository into Colab.")

print(f"Project root: {PROJECT_ROOT}")
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "requirements-colab.txt")]
)


## Environment check


In [ ]:
import hashlib
import random
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from tqdm.auto import tqdm

from gliner2_project.analysis import (
    compare_method_examples,
    confusion_pairs_frame,
    method_metrics_frame,
    per_label_delta_frame,
    per_label_metrics_frame,
    schema_disagreement_frame,
)
from gliner2_project.cluster_second_pass import (
    CLUSTER_SECOND_PASS_FROM_MEAN_CONFIDENCE,
    CLUSTER_SECOND_PASS_FROM_PLAIN,
    build_second_pass_plan,
    merge_second_pass_prediction,
    second_pass_summary_frame,
)
from gliner2_project.data_utils import ensure_label_text_column, stratified_subset, unique_label_texts
from gliner2_project.ensemble import MEAN_CONFIDENCE_ENSEMBLE, mean_confidence_ensemble
from gliner2_project.env_utils import print_environment_info
from gliner2_project.gliner2_wrappers import load_gliner2_model, predict_one_classification
from gliner2_project.hierarchical_gliner import (
    HIERARCHICAL_GLINER,
    build_coarse_group_schema,
    hierarchical_prediction_row,
    labels_for_hierarchical_second_stage,
    safety_group_union,
)
from gliner2_project.label_clusters import get_cluster_labels, get_cluster_name
from gliner2_project.label_descriptions import (
    DESCRIPTION_ALL_LABELS,
    build_contrastive_description_schema,
    build_description_schema,
    build_mixed_description_schema,
)
from gliner2_project.plotting import plot_metric_bar, plot_schema_disagreement_histogram, plot_top_label_improvements
from gliner2_project.schema_variants import (
    BANKING_REQUEST_LABEL,
    CUSTOMER_INTENT_LABEL,
    ENSEMBLE_INPUT_METHODS,
    PLAIN_LABEL,
    QUERY_ABOUT_LABEL,
    build_schema_variant,
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
RUN_START_UTC = datetime.now(timezone.utc).isoformat()
environment_info = print_environment_info()
print(f"Run started at: {RUN_START_UTC}")


## Load Banking77 test split only


In [ ]:
test_split = ensure_label_text_column(load_dataset(DATASET_NAME, split="test"))
test_examples = []
for idx, row in enumerate(test_split):
    item = dict(row)
    item.setdefault("example_id", idx)
    item["text"] = str(item["text"])
    item["label_text"] = str(item["label_text"])
    test_examples.append(item)

label_texts = unique_label_texts(test_examples)
print(f"Test examples: {len(test_examples)}")
print(f"Labels: {len(label_texts)}")
print("No Banking77 train split is loaded in this notebook.")


## Build evaluation subset


In [ ]:
if MODE == "smoke":
    eval_examples = list(test_examples[:SMOKE_N_EXAMPLES])
elif MODE == "small":
    eval_examples = stratified_subset(test_examples, per_label=SMALL_PER_LABEL, seed=SEED)
elif MODE == "full":
    eval_examples = list(test_examples) if FULL_USE_ALL_TEST else stratified_subset(test_examples, per_label=SMALL_PER_LABEL, seed=SEED)
else:
    raise ValueError(f"Unknown MODE: {MODE}")

example_ids = [example["example_id"] for example in eval_examples]
print(f"Mode: {MODE}")
print(f"Evaluated examples: {len(eval_examples)}")


## Load GLiNER2


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    print("WARNING: GPU is unavailable. GLiNER2 inference may be slow.")
else:
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")

model = load_gliner2_model(MODEL_NAME, device=DEVICE)
print("Model loaded.")


## Cache helpers


In [ ]:
def json_safe(value):
    try:
        json.dumps(value)
        return value
    except TypeError:
        return str(value)


def stable_hash(value):
    encoded = json.dumps(value, sort_keys=True, ensure_ascii=False, default=str).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()


def load_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def save_jsonl(rows, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False, default=json_safe) + "\n")


BASE_MANIFEST = {
    "project_name": PROJECT_NAME,
    "model_name": MODEL_NAME,
    "dataset_name": DATASET_NAME,
    "split": "test",
    "mode": MODE,
    "num_examples": len(eval_examples),
    "example_ids": example_ids,
    "seed": SEED,
}


def manifest_for(method_name, extra=None):
    manifest = dict(BASE_MANIFEST)
    manifest["method"] = method_name
    if extra:
        manifest.update(extra)
    manifest["manifest_hash"] = stable_hash({k: v for k, v in manifest.items() if k != "manifest_hash"})
    return manifest


def cache_manifest_matches(path, expected):
    path = Path(path)
    if not path.exists():
        return False
    try:
        cached = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return False
    return cached == expected


def load_method_cache(method_name, manifest):
    if FORCE_RERUN:
        return None
    pred_path = PREDICTIONS_DIR / f"{method_name}.jsonl"
    manifest_path = PREDICTIONS_DIR / f"{method_name}.manifest.json"
    if pred_path.exists() and cache_manifest_matches(manifest_path, manifest):
        print(f"Loading cached {method_name}")
        return load_jsonl(pred_path)
    return None


def save_method_cache(method_name, rows, manifest):
    save_jsonl(rows, PREDICTIONS_DIR / f"{method_name}.jsonl")
    (PREDICTIONS_DIR / f"{method_name}.manifest.json").write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False, default=json_safe),
        encoding="utf-8",
    )


## Run schema methods


In [ ]:
predictions_by_method = {}


def run_candidate_schema(method_name, candidate_labels, candidate_to_canonical, extra_manifest=None):
    manifest = manifest_for(method_name, extra_manifest)
    cached = load_method_cache(method_name, manifest)
    if cached is not None:
        predictions_by_method[method_name] = cached
        return cached

    rows = []
    for example in tqdm(eval_examples, desc=method_name):
        row = predict_one_classification(
            model=model,
            text=example["text"],
            candidate_labels=list(candidate_labels),
            candidate_to_canonical=dict(candidate_to_canonical),
            method_name=method_name,
            example_id=example["example_id"],
            gold_label=example["label_text"],
        )
        rows.append(row)
    save_method_cache(method_name, rows, manifest)
    predictions_by_method[method_name] = rows
    return rows


schema_methods_to_run = [PLAIN_LABEL]
if RUN_CLUSTER_SECOND_PASS or RUN_ORACLE_ANALYSIS or RUN_HIERARCHICAL_GLINER:
    schema_methods_to_run = list(dict.fromkeys(ENSEMBLE_INPUT_METHODS))

for method_name in schema_methods_to_run:
    candidates, mapping = build_schema_variant(label_texts, method_name)
    run_candidate_schema(
        method_name,
        candidates,
        mapping,
        extra_manifest={"schema_type": "deterministic_variant"},
    )

if RUN_DESCRIPTION_ALL_LABELS:
    candidates, mapping = build_description_schema(label_texts)
    run_candidate_schema(
        DESCRIPTION_ALL_LABELS,
        candidates,
        mapping,
        extra_manifest={"schema_type": "rule_descriptions"},
    )

def add_effective_latency_from_components(rows, component_methods):
    indexes = {
        method: {row.get("example_id"): row for row in predictions_by_method.get(method, [])}
        for method in component_methods
    }
    for row in rows:
        component_latencies = [
            indexes[method].get(row.get("example_id"), {}).get("latency_sec")
            for method in component_methods
        ]
        if all(value is not None for value in component_latencies):
            row["latency_sec"] = float(sum(float(value) for value in component_latencies))
    return rows


if all(method in predictions_by_method for method in ENSEMBLE_INPUT_METHODS):
    mean_rows = mean_confidence_ensemble(predictions_by_method)
    mean_rows = add_effective_latency_from_components(mean_rows, ENSEMBLE_INPUT_METHODS)
    for row in mean_rows:
        row["method"] = MEAN_CONFIDENCE_ENSEMBLE
    predictions_by_method[MEAN_CONFIDENCE_ENSEMBLE] = mean_rows
    save_jsonl(mean_rows, PREDICTIONS_DIR / f"{MEAN_CONFIDENCE_ENSEMBLE}.jsonl")


## Cluster second pass


In [ ]:
def run_cluster_second_pass(source_method, output_method):
    source_rows = predictions_by_method.get(source_method, [])
    manifest = manifest_for(
        output_method,
        {"source_method": source_method, "schema_type": "contrastive_cluster_second_pass", "latency_version": 2},
    )
    cached = load_method_cache(output_method, manifest)
    if cached is not None:
        predictions_by_method[output_method] = cached
        return cached

    output_rows = []
    for first_row, plan in tqdm(
        zip(source_rows, build_second_pass_plan(source_rows, source_method)),
        total=len(source_rows),
        desc=output_method,
    ):
        if not plan["run_second_pass"]:
            output_rows.append(
                merge_second_pass_prediction(
                    first_row,
                    None,
                    output_method,
                    cluster_name=plan["cluster_name"],
                )
            )
            continue
        cluster_labels = plan["cluster_labels"]
        candidates, mapping = build_contrastive_description_schema(
            cluster_labels,
            cluster_name=plan["cluster_name"],
        )
        second_row = predict_one_classification(
            model=model,
            text=first_row["text"],
            candidate_labels=candidates,
            candidate_to_canonical=mapping,
            method_name=f"{output_method}_detail",
            example_id=first_row["example_id"],
            gold_label=first_row["gold_label"],
        )
        output_rows.append(
            merge_second_pass_prediction(
                first_row,
                second_row,
                output_method,
                cluster_name=plan["cluster_name"],
            )
        )
    save_method_cache(output_method, output_rows, manifest)
    predictions_by_method[output_method] = output_rows
    return output_rows


if RUN_CLUSTER_SECOND_PASS:
    if PLAIN_LABEL in predictions_by_method:
        run_cluster_second_pass(PLAIN_LABEL, CLUSTER_SECOND_PASS_FROM_PLAIN)
    if MEAN_CONFIDENCE_ENSEMBLE in predictions_by_method:
        run_cluster_second_pass(MEAN_CONFIDENCE_ENSEMBLE, CLUSTER_SECOND_PASS_FROM_MEAN_CONFIDENCE)


## Hierarchical GLiNER


In [ ]:
def run_hierarchical_gliner():
    manifest = manifest_for(
        HIERARCHICAL_GLINER,
        {"schema_type": "coarse_to_fine_safety_union"},
    )
    cached = load_method_cache(HIERARCHICAL_GLINER, manifest)
    if cached is not None:
        predictions_by_method[HIERARCHICAL_GLINER] = cached
        return cached

    coarse_candidates, coarse_mapping = build_coarse_group_schema()
    coarse_rows = []
    output_rows = []
    plain_index = {row["example_id"]: row for row in predictions_by_method.get(PLAIN_LABEL, [])}
    schema_indexes = {
        method: {row["example_id"]: row for row in predictions_by_method.get(method, [])}
        for method in ENSEMBLE_INPUT_METHODS
    }

    for example in tqdm(eval_examples, desc=HIERARCHICAL_GLINER):
        base_row = plain_index.get(
            example["example_id"],
            {
                "example_id": example["example_id"],
                "text": example["text"],
                "gold_label": example["label_text"],
                "predicted_canonical": None,
                "latency_sec": 0.0,
            },
        )
        coarse_row = predict_one_classification(
            model=model,
            text=example["text"],
            candidate_labels=coarse_candidates,
            candidate_to_canonical=coarse_mapping,
            method_name="coarse_group_classifier",
            example_id=example["example_id"],
            gold_label=None,
        )
        coarse_rows.append(coarse_row)
        method_predictions = {
            method: schema_indexes.get(method, {}).get(example["example_id"], {}).get("predicted_canonical")
            for method in ENSEMBLE_INPUT_METHODS
        }
        groups_used = safety_group_union(
            first_pass_prediction=base_row.get("predicted_canonical"),
            coarse_group_prediction=coarse_row.get("predicted_canonical"),
            schema_variant_predictions=method_predictions,
        )
        candidate_labels = labels_for_hierarchical_second_stage(label_texts, groups_used)
        if not candidate_labels:
            candidate_labels = list(label_texts)
        fine_candidates, fine_mapping = build_mixed_description_schema(candidate_labels)
        fine_row = predict_one_classification(
            model=model,
            text=example["text"],
            candidate_labels=fine_candidates,
            candidate_to_canonical=fine_mapping,
            method_name="hierarchical_fine_classifier",
            example_id=example["example_id"],
            gold_label=example["label_text"],
        )
        output_rows.append(
            hierarchical_prediction_row(
                base_row=base_row,
                coarse_row=coarse_row,
                fine_row=fine_row,
                groups_used=groups_used,
                candidate_labels_used=candidate_labels,
            )
        )
    save_jsonl(coarse_rows, PREDICTIONS_DIR / "coarse_group_classifier.jsonl")
    save_method_cache(HIERARCHICAL_GLINER, output_rows, manifest)
    predictions_by_method[HIERARCHICAL_GLINER] = output_rows
    return output_rows


if RUN_HIERARCHICAL_GLINER:
    run_hierarchical_gliner()


## Analysis helpers


In [ ]:
def method_index(method):
    return {row.get("example_id"): row for row in predictions_by_method.get(method, [])}


def oracle_accuracy_frame(methods):
    available = [method for method in methods if method in predictions_by_method]
    if not available:
        return pd.DataFrame()
    indexes = {method: method_index(method) for method in available}
    first = available[0]
    rows = []
    for example_id, base_row in indexes[first].items():
        gold = base_row.get("gold_label")
        predictions = {
            method: indexes[method].get(example_id, {}).get("predicted_canonical")
            for method in available
        }
        rows.append(
            {
                "example_id": example_id,
                "gold_label": gold,
                "oracle_correct": any(pred == gold for pred in predictions.values()),
                "method_predictions": predictions,
            }
        )
    frame = pd.DataFrame(rows)
    return pd.DataFrame(
        [
            {
                "methods": ", ".join(available),
                "num_examples": int(len(frame)),
                "oracle_accuracy": float(frame["oracle_correct"].mean()) if not frame.empty else 0.0,
                "oracle_correct": int(frame["oracle_correct"].sum()) if not frame.empty else 0,
            }
        ]
    )


def per_cluster_accuracy(method_name):
    rows = predictions_by_method.get(method_name, [])
    frame = pd.DataFrame(rows)
    if frame.empty:
        return pd.DataFrame()
    frame["gold_cluster"] = frame["gold_label"].map(get_cluster_name).fillna("no_cluster")
    frame["is_correct"] = frame["gold_label"] == frame["predicted_canonical"]
    return (
        frame.groupby("gold_cluster")
        .agg(num_examples=("example_id", "count"), accuracy=("is_correct", "mean"))
        .reset_index()
        .assign(method=method_name)
    )


## Save tables and figures


In [ ]:
results_summary = method_metrics_frame(predictions_by_method)
results_summary.to_csv(TABLES_DIR / "results_summary.csv", index=False)
display(results_summary.sort_values("accuracy", ascending=False))

per_label = per_label_metrics_frame(predictions_by_method)
per_label.to_csv(TABLES_DIR / "per_label_metrics.csv", index=False)

delta_frames = []
for comparison in [
    DESCRIPTION_ALL_LABELS,
    CLUSTER_SECOND_PASS_FROM_PLAIN,
    CLUSTER_SECOND_PASS_FROM_MEAN_CONFIDENCE,
    HIERARCHICAL_GLINER,
]:
    if PLAIN_LABEL in predictions_by_method and comparison in predictions_by_method:
        delta_frames.append(per_label_delta_frame(per_label, PLAIN_LABEL, comparison))
per_label_delta = pd.concat(delta_frames, ignore_index=True) if delta_frames else pd.DataFrame()
per_label_delta.to_csv(TABLES_DIR / "per_label_delta_vs_plain.csv", index=False)

confusion_frames = []
for method, rows in predictions_by_method.items():
    pairs = confusion_pairs_frame(rows, top_n=50)
    if not pairs.empty:
        pairs["method"] = method
        confusion_frames.append(pairs)
confusions = pd.concat(confusion_frames, ignore_index=True) if confusion_frames else pd.DataFrame()
confusions.to_csv(TABLES_DIR / "confusion_pairs.csv", index=False)

second_pass_frames = []
for method in [CLUSTER_SECOND_PASS_FROM_PLAIN, CLUSTER_SECOND_PASS_FROM_MEAN_CONFIDENCE]:
    if method in predictions_by_method:
        frame = second_pass_summary_frame(predictions_by_method[method])
        frame["method"] = method
        second_pass_frames.append(frame)
second_pass_summary = pd.concat(second_pass_frames, ignore_index=True) if second_pass_frames else pd.DataFrame()
second_pass_summary.to_csv(TABLES_DIR / "second_pass_summary.csv", index=False)
display(second_pass_summary)

cluster_accuracy = pd.concat(
    [per_cluster_accuracy(method) for method in predictions_by_method],
    ignore_index=True,
) if predictions_by_method else pd.DataFrame()
cluster_accuracy.to_csv(TABLES_DIR / "per_cluster_accuracy.csv", index=False)

if RUN_ORACLE_ANALYSIS:
    oracle_methods = [
        PLAIN_LABEL,
        QUERY_ABOUT_LABEL,
        BANKING_REQUEST_LABEL,
        CUSTOMER_INTENT_LABEL,
        DESCRIPTION_ALL_LABELS,
        CLUSTER_SECOND_PASS_FROM_PLAIN,
        CLUSTER_SECOND_PASS_FROM_MEAN_CONFIDENCE,
        HIERARCHICAL_GLINER,
    ]
    oracle = oracle_accuracy_frame(oracle_methods)
else:
    oracle = pd.DataFrame()
oracle.to_csv(TABLES_DIR / "oracle_zero_shot_schema_variants.csv", index=False)
display(oracle)

disagreement_methods = [
    method for method in [
        PLAIN_LABEL,
        QUERY_ABOUT_LABEL,
        BANKING_REQUEST_LABEL,
        CUSTOMER_INTENT_LABEL,
        DESCRIPTION_ALL_LABELS,
        CLUSTER_SECOND_PASS_FROM_PLAIN,
        CLUSTER_SECOND_PASS_FROM_MEAN_CONFIDENCE,
        HIERARCHICAL_GLINER,
    ]
    if method in predictions_by_method
]
disagreement = schema_disagreement_frame(predictions_by_method, disagreement_methods)
disagreement.to_csv(TABLES_DIR / "zero_shot_schema_disagreement.csv", index=False)
if not disagreement.empty:
    disagreement_rate = float((~disagreement["all_agree"]).mean())
else:
    disagreement_rate = None
print("Zero-shot schema disagreement rate:", disagreement_rate)

if PLAIN_LABEL in predictions_by_method:
    for comparison in [
        DESCRIPTION_ALL_LABELS,
        CLUSTER_SECOND_PASS_FROM_PLAIN,
        CLUSTER_SECOND_PASS_FROM_MEAN_CONFIDENCE,
        HIERARCHICAL_GLINER,
    ]:
        if comparison in predictions_by_method:
            improved, degraded = compare_method_examples(predictions_by_method, PLAIN_LABEL, comparison)
            improved.to_csv(TABLES_DIR / f"improved_examples_{comparison}_vs_plain.csv", index=False)
            degraded.to_csv(TABLES_DIR / f"degraded_examples_{comparison}_vs_plain.csv", index=False)

if not results_summary.empty:
    plot_metric_bar(results_summary, "accuracy", FIGURES_DIR / "zero_shot_schema_accuracy.png", "Zero-shot Schema Accuracy")
    plot_metric_bar(results_summary, "macro_f1", FIGURES_DIR / "zero_shot_schema_macro_f1.png", "Zero-shot Schema Macro F1")
if not disagreement.empty:
    plot_schema_disagreement_histogram(disagreement, FIGURES_DIR / "zero_shot_schema_disagreement.png")
if not per_label_delta.empty:
    plot_top_label_improvements(per_label_delta, FIGURES_DIR / "top_label_improvements_zero_shot_schema.png")


## Save run manifest


In [ ]:
RUN_END_UTC = datetime.now(timezone.utc).isoformat()
manifest = {
    "project_name": PROJECT_NAME,
    "model_name": MODEL_NAME,
    "dataset_name": DATASET_NAME,
    "split": "test",
    "mode": MODE,
    "num_examples": len(eval_examples),
    "example_ids": example_ids,
    "seed": SEED,
    "device": DEVICE,
    "no_train_split_loaded": True,
    "run_description_all_labels": RUN_DESCRIPTION_ALL_LABELS,
    "run_cluster_second_pass": RUN_CLUSTER_SECOND_PASS,
    "run_hierarchical_gliner": RUN_HIERARCHICAL_GLINER,
    "run_oracle_analysis": RUN_ORACLE_ANALYSIS,
    "methods": list(predictions_by_method),
    "start_time_utc": RUN_START_UTC,
    "end_time_utc": RUN_END_UTC,
    "environment": environment_info,
}
(OUTPUT_PATH / "zero_shot_schema_run_manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False, default=json_safe),
    encoding="utf-8",
)
print(f"Artifacts saved under: {OUTPUT_PATH}")


## Compact summary for report


In [ ]:
print("=== Results summary ===")
display(results_summary.sort_values("accuracy", ascending=False))

print("\n=== Second-pass summary ===")
display(second_pass_summary)

print("\n=== Oracle zero-shot schema variants ===")
display(oracle)

print("\n=== Top per-label deltas vs plain_label ===")
display(per_label_delta.head(30) if not per_label_delta.empty else per_label_delta)

print("\nReporting caution:")
print("- This notebook does not load Banking77 train split.")
print("- Label descriptions are manually/rule-designed schema, not example annotations.")
print("- If schema choices were revised after viewing full test results, report the run as exploratory.")
print("- Do not claim SOTA or generalization beyond Banking77.")
